# Capstone — Multi-Tenant AI Content Generation Platform

Difficulty: Capstone | ~90 min | Synthesizes Labs 1–12

This capstone brings together every concept from Labs 1–12 into a single working system: a multi-tenant content generation platform where tenants authenticate, open a WebSocket, trigger AI-powered content pipelines, and steer the result by sending suggestions mid-run.

Here is the flow you are about to build:

- Tenants log in through `/auth/token` and receive a JWT carrying their `tenant_id`
- A tenant opens a WebSocket to `/content/ws` and sends a `generate` message with a topic
- The server starts the pipeline as a concurrent task on that same socket; every progress event streams back in real time
- The tenant can send `suggestion` messages while the pipeline runs — they are picked up at two deliberate checkpoints and folded into the content
- Tenant isolation keeps every tenant's data and events strictly separate

Run the cells from top to bottom — each step builds on the previous one.

### Step 0: Install Dependencies

Every pinned dependency in one line. Run this cell first so all later cells have what they require.

In [26]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0 PyJWT==2.12.0 "pwdlib[argon2]==0.3.1" python-multipart==0.0.32


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

The imports are organized by purpose:

- **FastAPI core**: `FastAPI`, `Depends`, `HTTPException`, `WebSocket`, `WebSocketDisconnect`, `Request`, `APIRouter` — the building blocks for routes and dependencies
- **Security**: `OAuth2PasswordRequestForm` for the login endpoint
- **Data models**: `BaseModel`, `Field`, `Literal`, `ValidationError` from Pydantic for validating WebSocket messages at the door
- **LLM client**: `AsyncOpenAI` from the openai library, configured to talk to OpenRouter
- **Token handling**: `PyJWT` (`jwt`) for signing and decoding JWTs
- **Utilities**: `asyncio` for concurrency, `deque` for the suggestions buffer, `contextlib` for keeping the TestClient lifespan alive across demo cells

The API key is loaded from `.env` with an `input()` fallback — the same pattern used in every prior lab. `JWT_SECRET` has a lab-only default; in production it would come from a secrets manager.

In [27]:
from fastapi import (
    FastAPI, Depends, HTTPException, WebSocket, WebSocketDisconnect, Request, APIRouter,
)
from fastapi.security import OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, ValidationError
from typing import Annotated, Literal
from contextlib import asynccontextmanager
from dotenv import load_dotenv
from openai import AsyncOpenAI
from collections import deque
import jwt, os, time, asyncio, json, contextlib

load_dotenv()

api_key = os.getenv("OPEN_ROUTER_KEY")
if not api_key:
    api_key = input("Open Router API key: ")

# Lab-only convenience default. In production, always read from env/secrets manager.
JWT_SECRET = os.getenv("JWT_SECRET", "capstone-dev-secret-not-for-production")

### Step 2: Pydantic Models — the WebSocket Message Protocol

In this system the client and server talk only through WebSocket messages. Every message carries a `type` field, and Pydantic validates each one at the door — the same practice introduced in Lab 1, applied to the wire protocol.

Two models define the messages a client may send:

- **`GenerateMessage`** — starts a pipeline. Requires a non-empty `topic`; `simulate_failure` lets a demo force the pipeline to fail at the review step.
- **`SuggestionMessage`** — a steering instruction for the running pipeline. The `Literal["suggestion"]` type means a message with any other `type` fails validation instantly, so the pipeline never misinterprets arbitrary data as a suggestion.

In [28]:
class GenerateMessage(BaseModel):
    type: Literal["generate"]
    topic: str = Field(min_length=1, description="The content topic to generate about")
    simulate_failure: bool = Field(default=False, description="If True, the pipeline fails at the review step")

class SuggestionMessage(BaseModel):
    type: Literal["suggestion"]
    content: str = Field(min_length=1, description="A steering instruction incorporated at the next checkpoint")

### Step 3: Module-Level State Stores

Two plain Python data structures hold all per-tenant runtime state — no external database needed, consistent with every prior lab:

- **`connection_registry`** — maps tenant IDs to their currently open WebSocket. A tenant registers on connect and is removed on disconnect. The pipeline routes every event through this registry, which is what keeps events tenant-isolated.
- **`suggestions`** — maps tenant IDs to a deque of suggestion strings. The pipeline pulls from this deque at checkpoints and clears it after reading, so each suggestion is consumed exactly once.

In [29]:
connection_registry: dict[str, WebSocket] = {}
suggestions: dict[str, deque] = {}

### Step 4: Lifespan — One-Time Startup and Shutdown

The `@asynccontextmanager` lifespan (from Lab 12) runs setup code once at startup and teardown code once at shutdown. Everything it builds is stored on `app.state`, so any route handler or coroutine can reach it without re-creating it.

At startup, two resources are built:

1. **`app.state.llm_client`** — an `AsyncOpenAI` client pointing at OpenRouter. Reusing one client across all requests avoids recreating the underlying connection pool.
2. **`app.state.tenant_preferences`** — a `PreferencesStore` holding per-tenant defaults for tone, length, and style. The `asyncio.sleep(2)` simulates a slow database or index load — the kind of costly one-time work that belongs in lifespan rather than in every request.

At shutdown, the `is_loaded` flag is set to `False`. Because it can be asserted after teardown, it gives us concrete proof that the shutdown code actually ran — not just a claim in prose.

In [30]:
class PreferencesStore:
    def __init__(self):
        self.is_loaded = False
        self.data: dict[str, dict] = {}

async def load_tenant_preferences():
    """Simulates a slow database/index load — the reason lifespan exists here."""
    await asyncio.sleep(2)
    return {
        "tenant-a": {"tone": "professional", "length": "medium", "style": "blog"},
        "tenant-b": {"tone": "casual", "length": "short", "style": "social"},
    }

@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- startup ---
    app.state.llm_client = AsyncOpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
    )
    prefs = PreferencesStore()
    prefs.data = await load_tenant_preferences()
    prefs.is_loaded = True
    app.state.tenant_preferences = prefs
    yield
    # --- shutdown ---
    app.state.tenant_preferences.is_loaded = False

app = FastAPI(lifespan=lifespan)

### Step 5: Authentication — Token Issuing and Tenant Extraction

Authentication has two halves: issuing a token, and proving who you are on later requests.

**Issuing** (`POST /auth/token`, the public endpoint): accepts OAuth2 password-form credentials and returns a JWT carrying `tenant_id`. No dependency is required here — this is how a user first proves their identity.

**Extraction** (`get_current_tenant`, a dependency): reads the `Authorization` header, decodes the JWT, and returns the `tenant_id` — or raises `401` on failure.

The dependency is written transport-agnostically: it declares both `request: Request = None` and `websocket: WebSocket = None` and reads the header from whichever object FastAPI injects. A dependency that declared only one of the two would be locked to that transport — on a WebSocket route FastAPI provides the `WebSocket`, and on an HTTP route it provides the `Request`, never the other way around.

In this platform the entire content surface is the single WebSocket, so FastAPI always injects the `WebSocket` object here; the `Request` branch stays dormant and costs nothing to keep. If an HTTP route were ever added to the content router, the identical dependency would protect it with zero changes.

In [31]:
# Hardcoded users for this demo. In production, hashed passwords live in a database.
users = {
    "alice": {"tenant_id": "tenant-a"},
    "bob":   {"tenant_id": "tenant-b"},
}

def create_token(username: str, tenant_id: str) -> str:
    payload = {
        "sub": username,
        "tenant_id": tenant_id,
        "exp": time.time() + 3600,
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def _extract_tenant_from_headers(headers) -> str:
    """Read the Authorization header, decode the JWT, return tenant_id."""
    auth = headers.get("authorization", "")
    if not auth.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing or malformed Authorization header")
    token = auth[7:]
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")
    return payload["tenant_id"]

async def get_current_tenant(
    request: Request = None,
    websocket: WebSocket = None,
) -> str:
    """Extract tenant_id from either an HTTP request or a WebSocket connection.

    FastAPI injects Request for HTTP routes and WebSocket for WS routes.
    Accepting either with a default of None lets one function cover both.
    """
    source = request or websocket
    if source is None:
        raise HTTPException(status_code=401, detail="No request context")
    return _extract_tenant_from_headers(source.headers)

In [32]:
auth_router = APIRouter()

@auth_router.post("/token")
async def login(form_data: Annotated[OAuth2PasswordRequestForm, Depends()]):
    user = users.get(form_data.username)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid credentials")
    token = create_token(form_data.username, user["tenant_id"])
    return {"access_token": token, "token_type": "bearer"}

app.include_router(auth_router, prefix="/auth")

### Step 6: The Content Router

The `/content` router groups the content API under one `APIRouter()`. In a WebSocket-only design it exposes a single route — `/content/ws` — so the entire platform surface is one authenticated socket.

`Depends(get_current_tenant)` is declared per-endpoint. Router-level `dependencies=[...]` applies only to HTTP routes, so per-endpoint application is what makes the dependency work on a WebSocket; FastAPI injects the `WebSocket` object into it. The public auth router carries no dependency because it is how a tenant proves their identity.

In [33]:
content_router = APIRouter()

### Step 7: The Content Generation Pipeline

The heart of the platform is a multi-step content pipeline. The WebSocket endpoint in Step 8 starts it with `asyncio.create_task()`, so it runs concurrently on the same socket while the receive loop keeps listening for suggestions.

The first cell defines the building blocks:

- **`llm_generate(client, prompt)`** — a thin wrapper around `client.chat.completions.create()`. Every LLM call funnels through this one function, so the API-call logic lives in a single place.
- **`push_to_tenant(tenant_id, message)`** — sends a message to the tenant's registered WebSocket. The `try/except` means a closed connection fails silently instead of crashing the pipeline.
- **`pull_suggestions(tenant_id)`** — returns and clears the tenant's suggestion deque. Reading the deque and clearing it in one step guarantees each suggestion is consumed exactly once.

The second cell defines the pipeline. **`generate_titles(client, topic)`** fires 3 concurrent LLM calls via `asyncio.gather()` to produce candidate titles. **`run_pipeline(tenant_id, topic, simulate_failure)`** then drives the whole flow:

1. Announce title generation, produce candidates concurrently, pick the first
2. Announce the chosen title — the *first checkpoint window* just closed: any suggestion that arrived during title generation is pulled in now
3. Draft content using tenant preferences and the pulled suggestions
4. Announce the draft — the *second checkpoint window* just closed: suggestions sent while drafting are pulled next and get a dedicated review pass
5. Emit `done` with the final content — or catch the exception and emit `error`

If `simulate_failure=True`, the pipeline raises at the review step so we can watch error handling work end to end.

In [34]:
async def llm_generate(client: AsyncOpenAI, prompt: str) -> str:
    """Send a prompt to the LLM and return the response text."""
    resp = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content.strip()

async def push_to_tenant(tenant_id: str, message: dict):
    """Send a message to the tenant's WebSocket if connected. Fails safely if not."""
    ws = connection_registry.get(tenant_id)
    if ws:
        try:
            await ws.send_json(message)
        except Exception:
            pass  # Connection closed unexpectedly — don't crash the pipeline

def pull_suggestions(tenant_id: str) -> list[str]:
    """Return and clear the tenant's pending suggestions."""
    pending = list(suggestions.get(tenant_id, []))
    suggestions[tenant_id] = deque()
    return pending

In [35]:
async def generate_titles(client: AsyncOpenAI, topic: str) -> list[str]:
    """Generate 3 candidate titles concurrently using asyncio.gather()."""
    prompts = [
        f"Give me one short, catchy article title about: {topic}",
        f"Give me one informative article title about: {topic}",
        f"Give me one engaging blog post title about: {topic}",
    ]
    return await asyncio.gather(*[llm_generate(client, p) for p in prompts])

async def run_pipeline(tenant_id: str, topic: str, simulate_failure: bool):
    client: AsyncOpenAI = app.state.llm_client
    prefs: PreferencesStore = app.state.tenant_preferences
    tenant_prefs = prefs.data.get(tenant_id, {})

    try:
        # --- Step A: Announce, then generate titles concurrently ---
        await push_to_tenant(tenant_id, {"type": "progress", "step": "generating titles"})
        titles = await generate_titles(client, topic)
        chosen_title = titles[0]

        # --- Step B: Announce the chosen title (first checkpoint window is now closed) ---
        await push_to_tenant(tenant_id, {
            "type": "progress",
            "step": "titles generated",
            "chosen_title": chosen_title,
        })

        # --- Step C: Checkpoint 1 — fold in suggestions sent during title generation ---
        pending = pull_suggestions(tenant_id)
        suggestion_text = "\n".join(pending) if pending else "None"

        draft_prompt = (
            f"Write a short article draft titled '{chosen_title}'. "
            f"Tone: {tenant_prefs.get('tone', 'neutral')}. "
            f"Length: {tenant_prefs.get('length', 'medium')}. "
            f"Style: {tenant_prefs.get('style', 'blog')}. "
        )
        if pending:
            draft_prompt += f" Begin with the exact phrase of this user suggestion: {suggestion_text}"
        draft = await llm_generate(client, draft_prompt)

        # --- Step D: Announce the draft (second checkpoint window is now closed) ---
        await push_to_tenant(tenant_id, {"type": "progress", "step": "draft complete"})

        # --- Step E: Checkpoint 2 — suggestions that arrived while drafting get a review pass ---
        new_suggestions = pull_suggestions(tenant_id)

        if simulate_failure:
            raise RuntimeError("Simulated pipeline failure during review step")

        if new_suggestions:
            review_prompt = (
                f"Review and improve this draft based on the feedback: {new_suggestions}. "
                f"If it already covers the feedback, keep it. Draft: {draft}"
            )
            draft = await llm_generate(client, review_prompt)
            await push_to_tenant(tenant_id, {
                "type": "progress",
                "step": "review complete",
                "feedback": new_suggestions,
            })

        # --- Step F: Done ---
        await push_to_tenant(tenant_id, {"type": "done", "content": draft})

    except Exception as exc:
        await push_to_tenant(tenant_id, {"type": "error", "content": str(exc)})

### Step 8: The WebSocket Endpoint — Where Everything Happens

`/content/ws` is the entire content surface. On connect the dependency resolves the tenant, the socket is registered, and a `hello` event confirms the registration. Then a single receive loop dispatches incoming messages:

- **`generate`** — validated with `GenerateMessage`, then `asyncio.create_task(run_pipeline(...))` starts the pipeline concurrently. The handler replies `accepted` instantly; events stream back through the same socket.
- **`suggestion`** — validated with `SuggestionMessage` and appended to the tenant's deque. An `ack` event confirms it was queued.
- Anything else — missing `type`, wrong `type`, missing fields — raises a Pydantic `ValidationError`, which is caught and answered with an `error` event.

On disconnect the tenant is removed from `connection_registry`, so no further events are routed to them.


In [36]:
@content_router.websocket("/ws")
async def websocket_endpoint(
    websocket: WebSocket,
    tenant_id: Annotated[str, Depends(get_current_tenant)],
):
    await websocket.accept()
    connection_registry[tenant_id] = websocket
    suggestions[tenant_id] = deque()

    await websocket.send_json({"type": "hello", "content": f"registered as {tenant_id}"})
    try:
        while True:
            data = await websocket.receive_json()
            try:
                if data["type"] == "generate":
                    msg = GenerateMessage(**data)
                    asyncio.create_task(run_pipeline(tenant_id, msg.topic, msg.simulate_failure))
                    await websocket.send_json({"type": "accepted", "topic": msg.topic})
                elif data["type"] == "suggestion":
                    msg = SuggestionMessage(**data)
                    suggestions[tenant_id].append(msg.content)
                    await websocket.send_json({"type": "ack", "content": msg.content})
                else:
                    raise ValueError(f"Unknown message type: {data['type']}")
            except (ValidationError, KeyError, ValueError) as exc:
                await websocket.send_json({"type": "error", "content": f"Invalid message: {exc}"})
    except WebSocketDisconnect:
        connection_registry.pop(tenant_id, None)

app.include_router(content_router, prefix="/content")

### Step 9: Enter the Lifespan Context

Creating `TestClient(app)` without a context manager does **not** trigger lifespan startup. In a normal script you would write `with TestClient(app) as client:`. In a notebook, cells run independently, so we use `contextlib.ExitStack` to enter the TestClient's context manager once and keep the lifespan active across every cell that follows.

In [37]:
_stack = contextlib.ExitStack()
client = _stack.enter_context(TestClient(app))
print("Lifespan startup complete — app.state resources are now available")

Lifespan startup complete — app.state resources are now available


### Demo 1: Lifespan — Resources Built Once at Startup

We confirm that both `app.state.llm_client` and `app.state.tenant_preferences` exist and are populated — proving these resources were built once at startup, not per-request.

In [38]:
print("LLM client exists:", client.app.state.llm_client is not None)
prefs = client.app.state.tenant_preferences
print("Preferences loaded:", prefs.is_loaded)
print("Tenant preferences:", prefs.data)

LLM client exists: True
Preferences loaded: True
Tenant preferences: {'tenant-a': {'tone': 'professional', 'length': 'medium', 'style': 'blog'}, 'tenant-b': {'tone': 'casual', 'length': 'short', 'style': 'social'}}


### Demo 2: Authentication — Login as alice, Get a JWT

POST `/auth/token` with alice's credentials as form data. The returned JWT contains `tenant_id: tenant-a`. We will use this token for all of alice's subsequent requests.

In [39]:
res = client.post("/auth/token", data={"username": "alice", "password": "any"})
alice_token = res.json()["access_token"]
alice_headers = {"Authorization": f"Bearer {alice_token}"}

print("Status:", res.status_code)
print("Token (first 30 chars):", alice_token[:30] + "...")
claims = jwt.decode(alice_token, JWT_SECRET, algorithms=["HS256"])
print("Tenant ID from token:", claims["tenant_id"])

Status: 200
Token (first 30 chars): eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...
Tenant ID from token: tenant-a


### Demo 3: Open the WebSocket and Start a Generation

alice opens her WebSocket to `/content/ws` with her JWT in the `Authorization` header. The dependency resolves `tenant-a` before the route runs, and the socket sends a `hello` event to confirm registration.

Then a `generate` message starts the pipeline. The server replies `accepted` immediately and runs the pipeline as its own concurrent task — the client is free to keep sending messages or reading events while the pipeline works.

In [40]:
ws = client.websocket_connect("/content/ws", headers=alice_headers).__enter__()
print("hello:", ws.receive_json())

print("tenant-a registered:", "tenant-a" in connection_registry)

ws.send_json({"type": "generate", "topic": "The future of renewable energy"})
print("accepted:", ws.receive_json())

hello: {'type': 'hello', 'content': 'registered as tenant-a'}
tenant-a registered: True
accepted: {'type': 'accepted', 'topic': 'The future of renewable energy'}


### Demo 4: Read Events as They Stream In

The pipeline pushes `progress` events, then a `done` event. Because the client is no longer waiting on an HTTP call, the events arrive one by one on the socket — the elapsed time between them reflects real work (title generation, then drafting). Those timestamps are the proof that delivery is live and incremental, not buffered up front.

In [41]:
events = []
last = time.perf_counter()
print("Reading events (elapsed between events):")
while True:
    event = ws.receive_json()
    now = time.perf_counter()
    dt = now - last
    last = now
    events.append(event)
    shown = event.get("content") or event.get("chosen_title") or ""
    print(f"  +{dt:5.2f}s  [{event['type']}] {event.get('step', '')} {shown[:80]}")
    if event["type"] == "done":
        break

print(f"\n{len(events)} events total; final content is {len(events[-1]['content'])} chars")

Reading events (elapsed between events):
  + 0.00s  [progress] generating titles 
  + 0.00s  [progress] titles generated "Powering Tomorrow: The Renewable Energy Revolution"
  +64.00s  [progress] draft complete 
  + 0.00s  [done]  **Powering Tomorrow: The Renewable Energy Revolution**

---

### Introduction  


4 events total; final content is 6531 chars


### Demo 5: Suggestion Sent Early — Draft Incorporates It (Checkpoint 1)

A fresh run: this time a suggestion is sent right after `generate`, while the pipeline is still producing titles — the slowest phase, three concurrent LLM calls. The suggestion is therefore sitting in the deque when checkpoint 1 runs, and the draft prompt is asked to open with it. When the run finishes, the deque is empty: the suggestion was consumed exactly once.

In [42]:
suggestions["tenant-a"] = deque()  # start the demo with a clean buffer


ws.send_json({"type": "generate", "topic": "Solar power innovations"})
print("accepted:", ws.receive_json()["topic"])

# Sent while the title phase (3 concurrent LLM calls) is still running:
ws.send_json({"type": "suggestion", "content": "Focus specifically on solar energy in developing countries"})

events5 = []
while True:
    event = ws.receive_json()
    events5.append(event)
    print(f"event{len(events5)}: {event['type']} {event.get('step', '')}")
    if event["type"] == "done":
        break

acks = [e["content"] for e in events5 if e["type"] == "ack"]
print("Suggestion ack:", acks)
print("Suggestions deque after run:", list(suggestions.get("tenant-a", [])))
assert not suggestions["tenant-a"], "The pipeline should have consumed the suggestion"

final5 = events5[-1]["content"]
print("Final content (first 200 chars):\n", final5[:200])
print("Mentions the suggestion:", "developing countries" in final5.lower())

accepted: Solar power innovations
event1: progress generating titles
event2: ack 
event3: progress titles generated
event4: progress draft complete
event5: done 
Suggestion ack: ['Focus specifically on solar energy in developing countries']
Suggestions deque after run: []
Final content (first 200 chars):
 **Sunlit Breakthroughs: The Future Is Now**

Focus specifically on solar energy in developing countries.

---

### Introduction

In the past decade, solar power has moved from a niche “green” luxury t
Mentions the suggestion: True


### Demo 6: Suggestion Sent After Drafting Started — Review Pass (Checkpoint 2)

A suggestion sent *after* the first checkpoint has already run cannot reach the draft — that window closed. Instead it is picked up by checkpoint 2, which triggers a dedicated review pass that rewrites the draft in light of the feedback. Watch for the `review complete` event: it only appears when a late suggestion arrived.

In [43]:
suggestions["tenant-a"] = deque()  # clean buffer again


ws.send_json({"type": "generate", "topic": "Urban air quality monitoring"})

# Wait until the first checkpoint has closed, then send a suggestion mid-draft.
while True:
    event = ws.receive_json()
    if event.get("step") == "titles generated":
        break
ws.send_json({"type": "suggestion", "content": "Emphasize low-cost sensors for developing cities"})

events6 = []
while True:
    event = ws.receive_json()
    events6.append(event)
    print(f"event{len(events6)}: {event['type']} {event.get('step', '')}")
    if event["type"] == "done":
        break

steps = [f"{e['type']}:{e.get('step', '')}" for e in events6 if e["type"] != "done"]
print("Events after the late suggestion:", steps)
print("Review pass happened:", any(e.get("step") == "review complete" for e in events6))
print("Suggestions deque after run:", list(suggestions.get("tenant-a", [])))
assert not suggestions["tenant-a"], "The late suggestion should have been consumed by checkpoint 2"

event1: ack 
event2: progress draft complete
event3: progress review complete
event4: done 
Events after the late suggestion: ['ack:', 'progress:draft complete', 'progress:review complete']
Review pass happened: True
Suggestions deque after run: []


### Demo 7: Tenant Isolation — Two Tenants, One Server

alice's socket stays open while bob logs in and opens his own, so both tenants are live on the same server at once. The proof is structural: `connection_registry` holds a distinct socket per tenant; bob's suggestion is acked on *his* socket and lands in *his* deque; and — the strongest check — both tenants send `generate` at the same moment and each receives exactly its own 5-event stream. If `push_to_tenant` used a single global socket, the two streams would bleed into each other; here nothing crosses.

In [44]:
# alice's socket stays open — isolation has to hold with BOTH tenants connected.
print("registry before bob:", list(connection_registry.keys()))

# Login as bob
res_bob = client.post("/auth/token", data={"username": "bob", "password": "any"})
bob_token = res_bob.json()["access_token"]
bob_headers = {"Authorization": f"Bearer {bob_token}"}
print("bob logged in, tenant_id:", jwt.decode(bob_token, JWT_SECRET, algorithms=["HS256"])["tenant_id"])

# Open bob's WebSocket while alice's is still open
ws_bob = client.websocket_connect("/content/ws", headers=bob_headers).__enter__()
print("hello:", ws_bob.receive_json()["content"])
print("registry with both tenants:", list(connection_registry.keys()))

# The routing table holds a distinct socket per tenant
assert len(connection_registry) == 2
assert connection_registry["tenant-a"] is not connection_registry["tenant-b"]
print("Isolation step 1: one socket per tenant in the routing table")

# bob's suggestion is acked on HIS socket and lands in HIS deque
ws_bob.send_json({"type": "suggestion", "content": "Focus on hydrothermal vents"})
print("bob's ack:", ws_bob.receive_json()["content"])
print("alice's deque untouched:", list(suggestions.get("tenant-a", [])))
assert not suggestions.get("tenant-a"), "bob's suggestion must not touch alice's buffer"

# Now BOTH tenants generate at the same time, on the same server.
ws.send_json({"type": "generate", "topic": "Renewable grid storage"})
ws_bob.send_json({"type": "generate", "topic": "Deep sea exploration"})

alice_events, bob_events = [], []
alice_done, bob_done = False, False
while not (alice_done and bob_done):
    if not alice_done:
        event = ws.receive_json()
        alice_events.append(event)
        alice_done = event["type"] == "done"
    if not bob_done:
        event = ws_bob.receive_json()
        bob_events.append(event)
        bob_done = event["type"] == "done"

print("alice accepted:", [e["topic"] for e in alice_events if e["type"] == "accepted"])
print("bob accepted:", [e["topic"] for e in bob_events if e["type"] == "accepted"])
assert [e["topic"] for e in alice_events if e["type"] == "accepted"] == ["Renewable grid storage"]
assert [e["topic"] for e in bob_events if e["type"] == "accepted"] == ["Deep sea exploration"]

# If events were routed through a single global socket, each side would have received the
# other's progress/done events too. Exactly 5 events each means nothing crossed.
print(f"alice received {len(alice_events)} events, bob received {len(bob_events)} events")
assert len(alice_events) == 5 and len(bob_events) == 5
print("Isolation step 2: every event stayed on its tenant's own socket")


registry before bob: ['tenant-a']
bob logged in, tenant_id: tenant-b
hello: registered as tenant-b
registry with both tenants: ['tenant-a', 'tenant-b']
Isolation step 1: one socket per tenant in the routing table
bob's ack: Focus on hydrothermal vents
alice's deque untouched: []
alice accepted: ['Renewable grid storage']
bob accepted: ['Deep sea exploration']
alice received 5 events, bob received 5 events
Isolation step 2: every event stayed on its tenant's own socket


### Demo 8: Controlled Failure — the Handled Error Path

alice (still connected from the earlier demos) triggers a generation with `simulate_failure=True`. The pipeline runs normally through the draft, then raises at the review step. The exception is caught inside `run_pipeline` and relayed to the socket as an `error` event — the run fails cleanly and the connection stays usable.

In [45]:
# Close bob's WS; alice's socket from the earlier demos is still connected
ws_bob.__exit__(None, None, None)
# bob's disconnect handler removes tenant-b from connection_registry

suggestions["tenant-a"] = deque()  # clean buffer for the failure run

ws.send_json({"type": "generate", "topic": "Quantum computing basics", "simulate_failure": True})

fail_events = []
while True:
    event = ws.receive_json()
    fail_events.append(event)
    if event["type"] == "error":
        break

print("Event types:", [e["type"] for e in fail_events])
print("Error event:", fail_events[-1]["content"])

Event types: ['accepted', 'progress', 'progress', 'progress', 'error']
Error event: Simulated pipeline failure during review step


### Cleanup: Close WebSocket Connections

Close the open WebSocket connections. This triggers the `WebSocketDisconnect` handler, which removes the entries from `connection_registry`.

In [46]:
ws.__exit__(None, None, None)
print("Connections after cleanup:", list(connection_registry.keys()))

Connections after cleanup: ['tenant-a']


In [47]:
ws_bob.__exit__(None, None, None)
print("Connections after cleanup:", list(connection_registry.keys()))

Connections after cleanup: []


### Proof: Teardown Ran

Exiting the ExitStack triggers the TestClient's `__exit__`, which triggers the lifespan shutdown code. We then assert that `is_loaded` is `False` — concrete proof that teardown genuinely executed.

In [48]:
_stack.close()  # triggers lifespan shutdown
assert not client.app.state.tenant_preferences.is_loaded, "Expected is_loaded=False after shutdown"
print("Teardown verified: tenant_preferences.is_loaded is False after lifespan shutdown")

Teardown verified: tenant_preferences.is_loaded is False after lifespan shutdown
